# Grandpa's own voice (Piper fine-tune)

Trains a **local, offline** Piper voice from the Achchuu recordings, so Grandpa speaks in Captain's voice without ElevenLabs.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

1. Run all (Runtime → Run all). The first time, the first cell shows a **Choose Files** button: pick `D:\GrandpaVoice\achchuu.zip`. It stays on Drive after that.
2. Every epoch saves to Drive, so if Colab disconnects, just **Run all** again and it continues.
3. When the last cell prints `DONE`, download `grandpa-achchuu.onnx` and `grandpa-achchuu.onnx.json` from Drive → GrandpaVoice/out.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, shutil, glob
DRIVE = '/content/drive/MyDrive/GrandpaVoice'
for d in ('ckpt', 'out', 'parts'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
PARTS = 20  # achchuu.zip split into 3.6 MB pieces (achchuu.zip.00 ... .19), or 1 for the whole zip
if not os.path.exists(f'{DRIVE}/achchuu.zip'):
    print('Click "Choose Files" and pick achchuu.zip (or the next achchuu.zip.NN part).')
    while len(glob.glob(f'{DRIVE}/parts/achchuu.zip*')) < PARTS:
        up = files.upload()
        for name in up:
            shutil.move(name, f'{DRIVE}/parts/{name}')
            if name == 'achchuu.zip':
                PARTS = 1
        print(len(glob.glob(f'{DRIVE}/parts/achchuu.zip*')), 'of', PARTS, 'uploaded')
    pieces = sorted(glob.glob(f'{DRIVE}/parts/achchuu.zip*'))
    with open(f'{DRIVE}/achchuu.zip', 'wb') as out:
        for piece in pieces:
            out.write(open(piece, 'rb').read())
print('dataset:', os.path.getsize(f'{DRIVE}/achchuu.zip') // 2**20, 'MB on Drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y espeak-ng libespeak-ng-dev > /dev/null
pip -q install scikit-build ninja cmake
cd /content
[ -d piper1-gpl ] || git clone -q https://github.com/OHF-Voice/piper1-gpl.git
cd piper1-gpl
pip -q install -e '.[train]'
./build_monotonic_align.sh > /dev/null
python3 setup.py build_ext --inplace > /content/build.log 2>&1 || { tail -20 /content/build.log; exit 1; }
python3 -c "import sys; sys.path.insert(0, 'src'); from piper import espeakbridge; print('installed, espeakbridge OK')"

In [ ]:
%%bash
set -e
mkdir -p /content/achchuu
[ -f /content/achchuu/metadata.csv ] || unzip -o -q /content/drive/MyDrive/GrandpaVoice/achchuu.zip -d /content/achchuu < /dev/null
python3 - <<'PY'
import os
rows = [l.rstrip('\n').split('|', 1) for l in open('/content/achchuu/metadata.csv', encoding='utf-8') if '|' in l]
keep = [(i, t) for i, t in rows if os.path.exists(f'/content/achchuu/wavs/{i}.wav')]
with open('/content/achchuu/train.csv', 'w', encoding='utf-8') as f:
    for i, t in keep:
        f.write(f'{i}.wav|{t}\n')
print(len(keep), 'clips ready')
PY

In [ ]:
# Base voice to fine-tune from (Lessac, medium). Downloaded once, kept on Drive.
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
import os, urllib.request, pathlib, torch
BASE = f'{DRIVE}/ckpt/lessac-medium.ckpt'
if not os.path.exists(BASE):
    url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
    urllib.request.urlretrieve(url, BASE)
ck = torch.load(BASE, map_location='cpu', weights_only=False)
if 'hyper_parameters' in ck:
    # The old piper's settings (sample_bytes, ...) are not options of piper1-gpl's trainer,
    # and its pathlib paths do not load under PyTorch 2.6's weights_only default.
    def fix(o):
        if isinstance(o, pathlib.PurePath):
            return str(o)
        if isinstance(o, dict):
            return {k: fix(v) for k, v in o.items()}
        if isinstance(o, list):
            return [fix(v) for v in o]
        return o
    ck = fix(ck)
    ck.pop('hyper_parameters', None)
    ck.pop('datamodule_hyper_parameters', None)
    torch.save(ck, BASE)
print(os.path.getsize(BASE) // 2**20, 'MB base checkpoint ready')

In [ ]:
# Resume from the newest checkpoint on Drive, else start from Lessac.
import glob
mine = sorted(glob.glob(f'{DRIVE}/ckpt/achchuu-*.ckpt'), key=os.path.getmtime)
START = mine[-1] if mine else BASE
EPOCHS = 120   # extra epochs on top of Lessac: ~6,600 steps, about 5-6 hours on a free T4
# The Lessac checkpoint is already at epoch 2164 and fine-tuning continues its count,
# so the stop point is 2164 + EPOCHS (resuming your own checkpoint keeps the same stop point).
MAX_EPOCHS = 2164 + EPOCHS
print('starting from', START)

In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train fit \
  --data.voice_name "grandpa-achchuu" \
  --data.csv_path /content/achchuu/train.csv \
  --data.audio_dir /content/achchuu/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "en-us" \
  --data.cache_dir /content/cache \
  --data.config_path {DRIVE}/out/config.json \
  --data.batch_size 16 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.callbacks+=ModelCheckpoint \
  --trainer.callbacks.dirpath {DRIVE}/ckpt \
  --trainer.callbacks.filename "achchuu-{{epoch:04d}}" \
  --trainer.callbacks.every_n_epochs 5 \
  --trainer.callbacks.save_top_k 2 \
  --trainer.callbacks.monitor epoch \
  --trainer.callbacks.mode max \
  --ckpt_path {START}

In [ ]:
# Export the newest checkpoint for Grandpa
import glob, shutil
last = sorted(glob.glob(f'{DRIVE}/ckpt/achchuu-*.ckpt'), key=os.path.getmtime)[-1]
print('exporting', last)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file {DRIVE}/out/grandpa-achchuu.onnx
shutil.copy(f'{DRIVE}/out/config.json', f'{DRIVE}/out/grandpa-achchuu.onnx.json')
!echo 'Hello Captain, this is Grandpa, speaking in your own voice, right here on your computer.' | python3 -m piper -m {DRIVE}/out/grandpa-achchuu.onnx -f {DRIVE}/out/test.wav
from IPython.display import Audio, display
display(Audio(f'{DRIVE}/out/test.wav'))
print('DONE: download grandpa-achchuu.onnx and grandpa-achchuu.onnx.json from Drive/GrandpaVoice/out')

## On your PC (PowerShell)

```powershell
Copy-Item $HOME\Downloads\grandpa-achchuu.onnx*  $HOME\.Grandpa\voices\
[Environment]::SetEnvironmentVariable('GRANDPA_PIPER_VOICE','grandpa-achchuu','User')
cd D:\Grandpa
uv run --extra avatar --extra piper --extra tamil-voice python -m grandpa avatar
```
The startup line shows `Piper (grandpa-achchuu)`. With the ElevenLabs key still set, ElevenLabs is used first and your own local voice is the fallback; to use only the local one, set `GRANDPA_AVATAR_VOICE=piper`.